In [16]:
import sys
import os

import bigframes.pandas as bpd
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import ndcg_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils import shuffle

import optuna
from optuna.integration import LightGBMPruningCallback
import lightgbm as lgb

import gcsfs
import gc

import json
import pickle

project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
if project_root not in sys.path:
    sys.path.append(project_root)

from config.config import BUCKET_NAME, source_loc, output_loc, config_loc

sys.path.insert(0, source_loc)
from utilities.utility_functions import split_data, select_features, get_bq_data_sample, pickle_and_stream_to_gcs

import warnings
warnings.filterwarnings('ignore')

In [2]:
best_trial_score = -1.0
best_trees_per_fold = [] #empty list to hold trees for each fold of best optuna trial

##### Read data into big frame and create sample

In [3]:
df_sample = get_bq_data_sample()

Full data shape: (18931851, 39)
Sample data shape: (1803821, 39)


##### Select features and split data

In [6]:
#Select only features an target column for training
df_features, feature_list, sort_list = select_features(df_sample)

In [7]:
#Perform split and only return train_val_df to save memory
train_val_df, _, _ = split_data(df_features)
train_val_df = train_val_df.reset_index(drop = True)

del df_features
gc.collect()

0

##### Define objective for Bayesian search. Only performing light hyperparameter tuning due to size of data and later ensemble optimization

In [8]:
#Set n_estimators to 2000 for safe cap since early stopping callback is used
#Set learning rate to 0.05 (half of default for smaller steps while also not going too low due to data size)
#Set objective to binary for binary classification

#Set evaluation metric to average precision (precision-recall AUC) due to class imbalance. ROC AUC is avoided due to the potential inflation
#of this metric due to true negatives. With average precision, True negatives are ignored and the focus is on minimizing false negatives and false positives.
#This aligns with the business case as we want to limit false negatives (predict re-orders correctly) while also minimizing the 
#amount of false negatives (predicting a reorder that was not re-ordered).

#Goal of this light hyperparameter tuning is to find optimal shape of trees rather trying to squeeze out fractional gains with a lower learning rate

#reg lambda is looked at for L2 regularization due to few features combining for majority of importance (found in eda feature screening)
#Note: reg lambda penalizes the model for splitting on dominant features over and over again. It limits the importance of dominant features by preventing trees
#      from over-relying on these dominant features.

#Avoid max depth and use num leaves due to leaf-wise growth for lgbm (Allow for assymetrical growth across branches)

#Explore feature fraction (create diversity in trees for regularization/generalization)

def objective(trial):
    global best_trial_score, best_trees_per_fold

        #Define trial hyperparameters
    params = {
            'objective': 'binary', 'metric': 'average_precision',
            'device': 'gpu', 'learning_rate': 0.05, 'n_estimators': 2000,
            'num_leaves': trial.suggest_int('num_leaves', 31, 255),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 0.9),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True)
            }
        
    sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
    
    #Create empty cv cores list to track cross validation scores
    cv_scores = []

    #Create array to track predictions
    trial_oof_preds = np.zeros(len(train_val_df))

    #Create list to track trees per fold
    trial_trees_per_fold = []
        
    for train_index, val_index in sgkf.split(train_val_df[feature_list], train_val_df['label_reordered'], groups=train_val_df['user_id']):
    
        print('Starting new fold...')

        #Create train and val data sets for current fold
        train_fold = train_val_df.iloc[train_index].copy()
        val_fold = train_val_df.iloc[val_index].copy()

        #Use the same method used for the rankers to track the original index. This ensures code consistency and acts as an insurance policy against index scrambling.
        val_fold['original_index'] = val_fold.index
    
        #create df for x_train, x_val, and numpy array for y_train, and y_val (Use arrays to remove index map and reduce memory overhead)
        x_train = train_fold[feature_list]
        y_train = train_fold['label_reordered'].to_numpy(dtype=np.int8)
        x_val = val_fold[feature_list]
        y_val = val_fold['label_reordered'].to_numpy(dtype=np.int8)

        print('x_train shape: ', x_train.shape)
        print('y_train shape: ', y_train.shape)
        print('x_val shape: ', x_val.shape)
        print('y_val shape: ', y_val.shape)

        #Create lgbm classifier model instance utilize params dictionary for hyperparameters
        lgbm_classifier = lgb.LGBMClassifier(**params)

        #Create pruning callback to kill poor performing trials early
        pruning_callback = LightGBMPruningCallback(trial, 'average_precision')

        #Fit the model
        #Utilize early stopping for regularization. Set to 50 rounds to balance with learning rate 0.05 
        
        lgbm_classifier.fit(x_train, y_train, eval_set=[(x_val, y_val)],
                        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False),
                        pruning_callback]
        )
        
        # Append validation ndcg@5 to cv scores 
        cv_scores.append(lgbm_classifier.best_score_['valid_0']['average_precision'])

        #Save predictions for this fold and map to original index. By the end of all folds there will be a prediction for all of train_val_df mapped back to the original index
        trial_oof_preds[val_fold['original_index']] = lgbm_classifier.predict_proba(x_val)[:, 1]

        #Append tree count to trial_trees_per_fold
        trial_trees_per_fold.append(lgbm_classifier.best_iteration_) 

        # Remove from memory to free up RAM due to data size
        del train_fold, val_fold, x_train, x_val, y_train, y_val, lgbm_classifier
        gc.collect()

    avg_cv_score = np.mean(cv_scores)

    #If this is the best trial so far, save the oof predictions locally 
    if avg_cv_score > best_trial_score:
        best_trial_score = avg_cv_score
        best_trees_per_fold = trial_trees_per_fold
       
        oof_df = train_val_df[['user_id', 'anchor_order_number', 'label_reordered']].copy()
        oof_df['lgbm_classifier_pred'] = trial_oof_preds
        oof_df.to_parquet(os.path.join(output_loc, "lgbm_classifier_oof_preds.parquet"))

    # Remove from memory to free up RAM due to data size
    del trial_oof_preds
    gc.collect() 
            
    return avg_cv_score

##### Perform Bayesian Search

In [9]:
#Initialize study. Set it to maximize the objective
study = optuna.create_study(direction='maximize')

#Optimize utilizing objective function
#Set trials = 25 to enable the study to wrong long enough to find optimized parameters while also balancing data size
study.optimize(objective, n_trials=3)

#Get optimal trees per fold
optimal_trees = int(np.mean(best_trees_per_fold))

#Create params and trees dictionary
params = {
        "best_params": study.best_params,
        "optimal_trees": optimal_trees,
        "best_trees_per_fold": best_trees_per_fold
        }

[I 2026-08-02 19:27:21,702] A new study created in memory with name: no-name-79b4eaab-3974-447c-9d67-b8e4175aef56


Starting new fold...
x_train shape:  (955982, 15)
y_train shape:  (955982,)
x_val shape:  (478034, 15)
y_val shape:  (478034,)
[LightGBM] [Warning] feature_fraction is set=0.851094579892784, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.851094579892784
[LightGBM] [Warning] feature_fraction is set=0.851094579892784, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.851094579892784
[LightGBM] [Info] Number of positive: 86634, number of negative: 869348
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3004
[LightGBM] [Info] Number of data points in the train set: 955982, number of used features: 15
[LightGBM] [Info] Using GPU Device: NVIDIA L4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 13 dense feature groups (14.59 MB) transferred to GPU in 0.012530 secs. 1 spar

[I 2026-08-02 19:27:45,291] Trial 0 finished with value: 0.41947777155640426 and parameters: {'num_leaves': 78, 'feature_fraction': 0.851094579892784, 'reg_lambda': 0.8717673387145757}. Best is trial 0 with value: 0.41947777155640426.


Starting new fold...
x_train shape:  (955982, 15)
y_train shape:  (955982,)
x_val shape:  (478034, 15)
y_val shape:  (478034,)
[LightGBM] [Warning] feature_fraction is set=0.7311143753377818, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7311143753377818
[LightGBM] [Warning] feature_fraction is set=0.7311143753377818, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7311143753377818
[LightGBM] [Info] Number of positive: 86634, number of negative: 869348
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3004
[LightGBM] [Info] Number of data points in the train set: 955982, number of used features: 15
[LightGBM] [Info] Using GPU Device: NVIDIA L4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 13 dense feature groups (14.59 MB) transferred to GPU in 0.009680 secs. 1 

[I 2026-08-02 19:28:10,199] Trial 1 finished with value: 0.41917522459818546 and parameters: {'num_leaves': 173, 'feature_fraction': 0.7311143753377818, 'reg_lambda': 0.08424502798419585}. Best is trial 0 with value: 0.41947777155640426.


Starting new fold...
x_train shape:  (955982, 15)
y_train shape:  (955982,)
x_val shape:  (478034, 15)
y_val shape:  (478034,)
[LightGBM] [Warning] feature_fraction is set=0.6676174617581812, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6676174617581812
[LightGBM] [Warning] feature_fraction is set=0.6676174617581812, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6676174617581812
[LightGBM] [Info] Number of positive: 86634, number of negative: 869348
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3004
[LightGBM] [Info] Number of data points in the train set: 955982, number of used features: 15
[LightGBM] [Info] Using GPU Device: NVIDIA L4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 13 dense feature groups (14.59 MB) transferred to GPU in 0.009613 secs. 1 

[I 2026-08-02 19:28:35,678] Trial 2 finished with value: 0.4195586108556148 and parameters: {'num_leaves': 67, 'feature_fraction': 0.6676174617581812, 'reg_lambda': 0.12822224377880914}. Best is trial 2 with value: 0.4195586108556148.


In [10]:
# Save the best hyperparameters locally
param_file_path = os.path.join(config_loc, "lgbm_classifier_best_params.json")
with open(param_file_path, "w") as f:
    json.dump(params, f)

params

{'best_params': {'num_leaves': 67,
  'feature_fraction': 0.6676174617581812,
  'reg_lambda': 0.12822224377880914},
 'optimal_trees': 105,
 'best_trees_per_fold': [91, 103, 123]}

##### Train model on full data and save to cloud storage bucket

In [11]:
#Train model on full data
train_full = train_val_df
x_full = train_full[feature_list].copy()
y_full = train_full['label_reordered'].to_numpy(dtype=np.int8)

In [12]:
#Create lgbm classifier instance (use optimal trees and best_params found from study)
lgbm_classifier_final = lgb.LGBMClassifier(objective='binary', metric='average_precision', device='gpu', learning_rate=0.05, n_estimators=optimal_trees, **study.best_params)

#Fit the model
lgbm_classifier_final.fit(x_full, y_full)

[LightGBM] [Warning] feature_fraction is set=0.6676174617581812, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6676174617581812
[LightGBM] [Warning] feature_fraction is set=0.6676174617581812, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6676174617581812
[LightGBM] [Info] Number of positive: 129952, number of negative: 1304064
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3025
[LightGBM] [Info] Number of data points in the train set: 1434016, number of used features: 15
[LightGBM] [Info] Using GPU Device: NVIDIA L4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 13 dense feature groups (21.88 MB) transferred to GPU in 0.013911 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.090621 -> initscore=-2.306076
[LightGBM] [Info] Star

,boosting_type,'gbdt'
,num_leaves,67
,max_depth,-1
,learning_rate,0.05
,n_estimators,105
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [17]:
#Save final lgbm classifier to storage
pickle_and_stream_to_gcs(lgbm_classifier_final, BUCKET_NAME, 'lgbm_classifier_base_model.pkl')

🎉 Success! lgbm_classifier_base_model.pkl successfully streamed to Cloud Storage.


True